
# Multi-chain MCMC speedup via ``jax.vmap``

Demonstrates the ``n_chains`` knob on the gradient-based MCMC backends.
Each chain shares the cached step size and mass matrix (so this is only
meaningful on the *second* call against the same model), then ``jax.vmap``
dispatches the chains in parallel across XLA SIMD lanes (CPU) or
accelerator cores (GPU/TPU).

The single-chain wall is the per-leapfrog gradient throughput × number
of leapfrogs per iteration × ``n_samples``. The 4-chain wall is roughly
the same — you get ~4× more samples at the same wall, until the CPU/GPU
arithmetic ceiling.

Run::

    python examples/inference/plot_multichain_speedup.py

Output: ``plot_multichain_speedup.png`` (per-sample wall time, single

chain vs four chains, across NUTS / HMC / dynamic HMC / GHMC / MCLMC /
adjusted MCLMC / ray tracing).


In [ ]:
from __future__ import annotations

import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import time
from pathlib import Path

os.environ.setdefault("JAX_PLATFORMS", "cpu")

import jax
import matplotlib.pyplot as plt
import numpy as np

from tengri import (
    FIXED,
    FREE,
    Fixed,
    Observation,
    Photometry,
    SEDModel,
    Uniform,
    WavePrecomp,
    builders,
    generate_mock,
    load_ssp_data,
    plot,
)
from tengri.inference.fitter import Fitter

plot.setup_style()
HERE = Path(__file__).parent


def build_quickstart_model():
    """The 6-band quickstart model — fast enough to bench all backends."""
    ssp_path = HERE.parent.parent / "notebooks" / "data" / "fsps_prsc_miles_chabrier.h5"
    ssp = load_ssp_data(str(ssp_path))
    obs = Observation(
        photometry=Photometry.from_names(
            ["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z", "wise_w1"]
        )
    )
    sed_model = SEDModel.build(
        ssp_data=ssp,
        observation=obs,
        approx=WavePrecomp(),
        sfh=builders.sfh.tsnorm(defaults=FREE),
        dust=builders.dust.two_component(
            defaults=FIXED, law_bc="calzetti", tau_bc=Uniform(0.0, 1.0)
        ),
        neb=builders.neb.none(),
        redshift=Fixed(0.05),
    )
    return sed_model, obs


# Raytrace uses n_steps; the rest use n_samples. Wire the right kwarg.
_SAMPLE_COUNT_KWARG = {"mcmc_raytrace": "n_steps"}


def bench(method: str, model, data, noise, key, n_samples: int, n_chains: int) -> float:
    """Time a single Fitter.run() and return wall seconds."""
    kw = {_SAMPLE_COUNT_KWARG.get(method, "n_samples"): n_samples}
    t0 = time.perf_counter()
    Fitter(model, data, noise, data_type="photometry").run(
        method=method, key=key, n_chains=n_chains, verbose=False, **kw
    )
    return time.perf_counter() - t0


def main() -> None:
    sed_model, _obs = build_quickstart_model()
    key = jax.random.PRNGKey(7)
    truth = sed_model.spec.sample(key)
    mock = generate_mock(sed_model, truth, key=key, snr=30.0)
    flux_obs = np.asarray(mock["flux_obs"])
    noise = np.asarray(mock["noise"])

    methods = [
        "mcmc_nuts",
        "mcmc_hmc",
        "mcmc_dynamic_hmc",
        "mcmc_ghmc",
        "mcmc_mclmc",
        "mcmc_adjusted_mclmc",
        "mcmc_raytrace",
    ]
    n_samples = 200
    n_chains_target = 4

    results: list[dict] = []
    h1 = f"1×{n_samples}"
    hN = f"{n_chains_target}×{n_samples}"
    print(f"{'method':<24}{h1:>12}{hN:>12}{'per-sample×':>14}")
    print("-" * 64)

    for method in methods:
        # Pre-warm: populate adaptation cache + JIT (sub-sample throwaway fit).
        warmup_kw = {_SAMPLE_COUNT_KWARG.get(method, "n_samples"): 10}
        try:
            Fitter(sed_model, flux_obs, noise, data_type="photometry").run(
                method=method, key=key, verbose=False, **warmup_kw
            )
        except Exception as exc:
            print(f"{method:<24}skipped ({type(exc).__name__}: {exc})")
            continue

        t1 = bench(method, sed_model, flux_obs, noise, key, n_samples, n_chains=1)
        tN = bench(method, sed_model, flux_obs, noise, key, n_samples, n_chains=n_chains_target)
        per_sample_speedup = n_chains_target * t1 / tN
        results.append(dict(method=method, t1=t1, tN=tN, speedup=per_sample_speedup))
        print(f"{method:<24}{t1:>11.2f}s{tN:>11.2f}s{per_sample_speedup:>13.2f}×")

    # Plot
    labels = [r["method"].replace("mcmc_", "") for r in results]
    t1 = np.array([r["t1"] for r in results])
    tN = np.array([r["tN"] for r in results])

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    x = np.arange(len(labels))

    # Left panel: wall time
    bar_w = 0.38
    axes[0].bar(x - bar_w / 2, t1, bar_w, label=f"1 chain × {n_samples}", color="#888")
    axes[0].bar(
        x + bar_w / 2,
        tN,
        bar_w,
        label=f"{n_chains_target} chains × {n_samples}",
        color="#3a76d9",
    )
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=20, ha="right")
    axes[0].set_ylabel("wall time [s]")
    axes[0].set_title("warm-cache sampling wall")
    axes[0].legend(frameon=False, fontsize=9)

    # Right panel: per-sample speedup
    speed = np.array([r["speedup"] for r in results])
    axes[1].bar(x, speed, color="#c3372a")
    axes[1].axhline(
        n_chains_target, color="0.4", ls="--", lw=0.8, label=f"ideal ({n_chains_target}×)"
    )
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=20, ha="right")
    axes[1].set_ylabel(r"per-sample speedup  $n_{\rm chains}\cdot t_1 / t_N$")
    axes[1].set_title(f"{n_chains_target} chains via vmap")
    axes[1].set_ylim(0, n_chains_target * 1.1)
    axes[1].legend(frameon=False, fontsize=9, loc="lower right")

    fig.suptitle(
        "MCMC backends: n_chains × jax.vmap parallelism\n"
        "6-band SDSS+W1 photometry, 7 free params, modified-blackbody dust IR",
        fontsize=10,
    )
    fig.tight_layout()
    out = HERE / "plot_multichain_speedup.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    print(f"\nsaved {out}")


if __name__ == "__main__":
    main()